# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshithavalli1006-spec/FlyRank--AI-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

Observation: The traffic fields are strongly right-skewed. Impressions_90d has a median of 731 but a 95th percentile of about 22,296, while sessions_90d has a median of 7 and a 95th percentile of 166. Word count and average position also have wider upper tails. This means a small number of pages have much larger values than most pages, so averages alone can be misleading when auditing signals.

In [3]:
!git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 299, done.
remote: Counting objects: 100% (193/193), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 299 (delta 130), reused 98 (delta 98), pack-reused 106 (from 1)
Receiving objects: 100% (299/299), 1.88 MiB | 4.07 MiB/s, done.
Resolving deltas: 100% (161/161), done.


In [4]:
import pandas as pd
import numpy as np
import os

# Load dataset
df = pd.read_csv(
    "flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"
)

print("Shape:", df.shape)

fields = [
    "impressions_90d",
    "sessions_90d",
    "word_count",
    "avg_position"
]

print("\nDistribution summary:")
print(df[fields].describe().T)

print("\nMedian vs 95th percentile:")
for col in fields:
    print(
        col,
        "median =", df[col].median(),
        "p95 =", df[col].quantile(0.95)
    )


Shape: (30000, 44)

Distribution summary:
                   count         mean           std  min     25%     50%  \
impressions_90d  30000.0  5200.366300  16838.019547  1.0    81.0   731.0   
sessions_90d     30000.0    37.066633    107.069131  1.0     2.0     7.0   
word_count       22301.0  3107.760325   1452.382598  8.0  2413.0  2877.0   
avg_position     30000.0    16.342380     15.216790  0.0     6.2    10.8   

                     75%       max  
impressions_90d  3615.25  517715.0  
sessions_90d       27.00    4345.0  
word_count       3666.00    9546.0  
avg_position       22.30     245.0  

Median vs 95th percentile:
impressions_90d median = 731.0 p95 = 22996.499999999993
sessions_90d median = 7.0 p95 = 166.0
word_count median = 2877.0 p95 = 6173.0
avg_position median = 10.8 p95 = 48.2


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Signal test #1 — CONFIRMED: Longer pages do not show a consistently increasing median impressions pattern across all buckets, so this signal is not strongly supported. I treat it as MIXED rather than confirmed.

Signal test #2 — CONFIRMED: Pages with better average positions have higher median impressions (1,249 at median position 5 vs. 609.5 at median position 33.95). This supports the assumption that stronger search position is associated with greater visibility.

Signal test #3 — MIXED: Older pages do not consistently have fewer impressions. Median impressions fall from 768 to 601 but rise again to 733 in the oldest bucket, so the assumption is MIXED.

In [5]:
# Signal tests: use log-transformed traffic because the fields are heavy-tailed

# Test 1: Longer pages get more impressions
test1 = (
    df.assign(
        word_bucket=pd.qcut(df["word_count"], 4, duplicates="drop")
    )
    .groupby("word_bucket", observed=True)
    .agg(
        n=("impressions_90d", "size"),
        median_impressions=("impressions_90d", "median"),
        median_word_count=("word_count", "median")
    )
    .reset_index()
)

print("TEST 1: Longer pages get more impressions")
print(test1)

# Test 2: Better average position gets more impressions
test2 = (
    df[df["avg_position"] > 0]
    .assign(
        position_bucket=pd.qcut(
            df[df["avg_position"] > 0]["avg_position"],
            4,
            duplicates="drop"
        )
    )
    .groupby("position_bucket", observed=True)
    .agg(
        n=("impressions_90d", "size"),
        median_impressions=("impressions_90d", "median"),
        median_position=("avg_position", "median")
    )
    .reset_index()
)

print("\nTEST 2: Better-positioned pages get more impressions")
print(test2)

# Test 3: Older pages have fewer impressions
test3 = (
    df.assign(
        age_bucket=pd.qcut(df["content_age_days"], 4, duplicates="drop")
    )
    .groupby("age_bucket", observed=True)
    .agg(
        n=("impressions_90d", "size"),
        median_impressions=("impressions_90d", "median"),
        median_age=("content_age_days", "median")
    )
    .reset_index()
)

print("\nTEST 3: Older pages get fewer impressions")
print(test3)


TEST 1: Longer pages get more impressions
        word_bucket     n  median_impressions  median_word_count
0   (7.999, 2413.0]  5576                91.0             1450.0
1  (2413.0, 2877.0]  5586              1096.5             2688.0
2  (2877.0, 3666.0]  5566               889.0             3121.0
3  (3666.0, 9546.0]  5573              1495.0             4896.0

TEST 2: Better-positioned pages get more impressions
  position_bucket     n  median_impressions  median_position
0    (0.099, 6.7]  7412              1249.0             5.00
1     (6.7, 11.4]  7076               938.5             8.50
2    (11.4, 22.9]  7115               843.0            16.00
3   (22.9, 245.0]  7192               609.5            33.95

TEST 3: Older pages get fewer impressions
        age_bucket     n  median_impressions  median_age
0  (89.999, 132.0]  7518               768.0       111.0
1   (132.0, 236.0]  8128               796.5       174.0
2   (236.0, 333.0]  6917               601.0       302.0
3  

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [9]:
flag_test = (
    df.groupby("provider_used", dropna=False)
      .agg(
          n=("impressions_90d", "size"),
          median_impressions=("impressions_90d", "median"),
          median_sessions=("sessions_90d", "median")
      )
      .sort_values("median_impressions", ascending=False)
      .reset_index()
)

print("Flag-linked test: provider_used")
print(flag_test)

Flag-linked test: provider_used
  provider_used      n  median_impressions  median_sessions
0        google   7364               902.5              7.0
1           NaN  21438               726.0              7.0
2        openai   1198                67.5             19.0


Flag-linked test — MIXED: The observed median impressions differ substantially by provider_used: Google has 902.5 median impressions, OpenAI has 67.5, and missing provider values have 726. The data therefore shows a strong difference between groups, but the missing-provider group is large, so I would treat the flag assumption as MIXED rather than confirmed. This is an observed association, not proof that the provider itself causes higher visibility.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

What this means in practice: The strongest observed signal is search position, while page length and content age show mixed relationships with impressions. The content team should prioritize pages with strong visibility and good search positions, but avoid relying on a single signal when deciding which pages to refresh. Provider differences should also be treated cautiously because many rows have missing provider information.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.